In [46]:
import numpy as np
import spacy
import pandas as pd

#Motto

1) All companies mentioned in a text
2) All stocks referenced in a text
3) All indexes mentioned in a text

In [47]:
data=pd.read_csv("/content/stocks.tsv",sep='\t')
data.head(10)

,Symbol,CompanyName,Industry,MarketCap
0,A,Agilent Technologies,Life Sciences Tools & Services,53.65B
1,AA,Alcoa,Metals & Mining,9.25B
2,AAC,Ares Acquisition,Shell Companies,1.22B
3,AACG,ATA Creativity Global,Diversified Consumer Services,90.35M
4,AADI,Aadi Bioscience,Pharmaceuticals,104.85M
5,AAIC,Arlington Asset Investment,Mortgage Real Estate Investment Trus...,120.92M
6,AAL,American Airlines,Airlines,12.27B
7,AAMC,Altisource Asset Management,Real Estate Management & Development,57.74M
8,AAME,Atlantic American,Insurance,80.01M
9,AAN,The Aaron's Company,Specialty Retail,857.00M


In [48]:
symbols=data.Symbol.tolist()
companies=data.CompanyName.tolist()
len(symbols)
print (symbols[0])
print (companies[0])

A
Agilent Technologies


In [49]:
data.info()
#we do have null characters

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5879 entries, 0 to 5878
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Symbol       5879 non-null   object
 1   CompanyName  5879 non-null   object
 2   Industry     5870 non-null   object
 3   MarketCap    5879 non-null   object
dtypes: object(4)
memory usage: 183.8+ KB


In [50]:
data2=pd.read_csv("/content/indexes.tsv",sep='\t')
data2.head(10)

,IndexName,IndexSymbol
0,Dow Jones Industrial Average,DJIA
1,Dow Jones Transportation Average,DJT
2,Dow Jones Utility Average Index,DJU
3,NASDAQ 100 Index (NASDAQ Calculation),NDX
4,NASDAQ Composite Index,COMP
5,NYSE Composite Index,NYA
6,S&P 500 Index,SPX
7,S&P 400 Mid Cap Index,MID
8,S&P 100 Index,OEX
9,NASDAQ Computer Index,IXCO


In [51]:
indexes=data2.IndexName.tolist()
index_symbols=data2.IndexSymbol.tolist()


In [52]:
data3=pd.read_csv("/content/stock_exchanges.tsv",sep='\t')
data3.head(10)

,BloombergExchangeCode,BloombergCompositeCode,Country,Description,ISOMIC,Google Prefix,EODcode,NumStocks
0,AF,AR,Argentina,Bolsa de Comercio de Buenos Aires,XBUE,NaN,BA,12
1,AO,AU,Australia,National Stock Exchange of Australia,XNEC,NaN,NaN,1
2,AT,AU,Australia,Asx - All Markets,XASX,ASX,AU,875
3,AV,NaN,Austria,Wiener Boerse Ag,XWBO,VIE,VI,38
4,BI,NaN,Bahrain,Bahrain Bourse,XBAH,NaN,NaN,4
5,BD,NaN,Bangladesh,Dhaka Stock Exchange Ltd,XDHA,NaN,NaN,18
6,BB,NaN,Belgium,Nyse Euronext - Euronext Brussels,XBRU,EBR,BR,75
7,BS,BZ,Brazil,XBSP,BVMF,BVMF,SA,369
8,CF,CN,Canada,Canadian Securities Exchange,XCNQ,NaN,CN,59
9,CT,CN,Canada,Toronto Stock Exchange,XTSE,TSE,TO,520


In [53]:
exchanges=data3.ISOMIC.tolist()+data3["Google Prefix"].tolist()
descs=data3.Description.tolist()

In [54]:
stops = ["two"]
nlp = spacy.blank("en")
ruler = nlp.add_pipe("entity_ruler")
patterns = []
letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
#List of Entities and Patterns
for symbol in symbols:
    patterns.append({"label": "STOCK", "pattern": symbol})
    for l in letters:
        patterns.append({"label": "STOCK", "pattern": symbol+f".{l}"})
        #Useful for exchange-specific tickers


for company in companies:
    if company not in stops: #Skip companies that are inside the stops list.
        patterns.append({"label": "COMPANY", "pattern": company})
        words = company.split()
        if len(words) > 1:
            new = " ".join(words[:2])  #Take only the FIRST TWO WORDS.
            patterns.append({"label": "COMPANY", "pattern": new}) #Add this shortened version as another matching pattern.

for index in indexes:
    patterns.append({"label": "INDEX", "pattern": index})
    versions = []
    words = index.split()
    caps = []
    for word in words:
        word = word.lower().capitalize()
        caps.append(word)
    versions.append(" ".join(caps)) # Properly capitalized full name
    versions.append(words[0]) #First word only
    versions.append(caps[0]) #first word only capatalised
    versions.append(" ".join(caps[:2])) #First two capitalized words
    versions.append(" ".join(words[:2])) #First two original words

    #now versions would be added as patterns

    for version in versions:
        if version != "NYSE":
            patterns.append({"label": "INDEX", "pattern": version})

'''
    So finally these patterns get added:

Dow Jones Industrial Average
dow
Dow
Dow Jones
dow jones

'''

for symbol in index_symbols:
    patterns.append({"label": "INDEX", "pattern": symbol})


for d in descs:
    patterns.append({"label": "STOCK_EXCHANGE", "pattern": d})
for e in exchanges:
    patterns.append({"label": "STOCK_EXCHANGE", "pattern": e})


ruler.add_patterns(patterns)



print (len(patterns))

169694


In [55]:
print(patterns[0:5])

[{'label': 'STOCK', 'pattern': 'A'}, {'label': 'STOCK', 'pattern': 'A.A'}, {'label': 'STOCK', 'pattern': 'A.B'}, {'label': 'STOCK', 'pattern': 'A.C'}, {'label': 'STOCK', 'pattern': 'A.D'}]


In [56]:
#source: https://www.reuters.com/business/futures-rise-after-biden-xi-call-oil-bounce-2021-09-10/
text = '''
Sept 10 (Reuters) - Wall Street's main indexes were subdued on Friday as signs of higher inflation and a drop in Apple shares following an unfavorable court ruling offset expectations of an easing in U.S.-China tensions.

Data earlier in the day showed U.S. producer prices rose solidly in August, leading to the biggest annual gain in nearly 11 years and indicating that high inflation was likely to persist as the pandemic pressures supply chains. read more .

"Today's data on wholesale prices should be eye-opening for the Federal Reserve, as inflation pressures still don't appear to be easing and will likely continue to be felt by the consumer in the coming months," said Charlie Ripley, senior investment strategist for Allianz Investment Management.

Apple Inc (AAPL.O) fell 2.7% following a U.S. court ruling in "Fortnite" creator Epic Games' antitrust lawsuit that stroke down some of the iPhone maker's restrictions on how developers can collect payments in apps.


Sponsored by Advertising Partner
Sponsored Video
Watch to learn more
Report ad
Apple shares were set for their worst single-day fall since May this year, weighing on the Nasdaq (.IXIC) and the S&P 500 technology sub-index (.SPLRCT), which fell 0.1%.

Sentiment also took a hit from Cleveland Federal Reserve Bank President Loretta Mester's comments that she would still like the central bank to begin tapering asset purchases this year despite the weak August jobs report. read more

Investors have paid keen attention to the labor market and data hinting towards higher inflation recently for hints on a timeline for the Federal Reserve to begin tapering its massive bond-buying program.

The S&P 500 has risen around 19% so far this year on support from dovish central bank policies and re-opening optimism, but concerns over rising coronavirus infections and accelerating inflation have lately stalled its advance.


Report ad
The three main U.S. indexes got some support on Friday from news of a phone call between U.S. President Joe Biden and Chinese leader Xi Jinping that was taken as a positive sign which could bring a thaw in ties between the world's two most important trading partners.

At 1:01 p.m. ET, the Dow Jones Industrial Average (.DJI) was up 12.24 points, or 0.04%, at 34,891.62, the S&P 500 (.SPX) was up 2.83 points, or 0.06%, at 4,496.11, and the Nasdaq Composite (.IXIC) was up 12.85 points, or 0.08%, at 15,261.11.

Six of the eleven S&P 500 sub-indexes gained, with energy (.SPNY), materials (.SPLRCM) and consumer discretionary stocks (.SPLRCD) rising the most.

U.S.-listed Chinese e-commerce companies Alibaba and JD.com , music streaming company Tencent Music (TME.N) and electric car maker Nio Inc (NIO.N) all gained between 0.7% and 1.4%


Report ad
Grocer Kroger Co (KR.N) dropped 7.1% after it said global supply chain disruptions, freight costs, discounts and wastage would hit its profit margins.

Advancing issues outnumbered decliners by a 1.12-to-1 ratio on the NYSE and by a 1.02-to-1 ratio on the Nasdaq.

The S&P index recorded 14 new 52-week highs and three new lows, while the Nasdaq recorded 49 new highs and 38 new lows.
'''

In [57]:
doc=nlp(text)
for ents in doc.ents:
  print(ents.text,ents.label_)

Apple COMPANY
Apple COMPANY
AAPL.O STOCK
Apple COMPANY
Nasdaq COMPANY
S&P 500 INDEX
S&P 500 INDEX
ET STOCK
Dow Jones Industrial Average INDEX
S&P 500 INDEX
Nasdaq Composite INDEX
S&P 500 INDEX
JD.com COMPANY
Tencent Music COMPANY
TME.N STOCK
NIO.N STOCK
Kroger COMPANY
KR.N STOCK
NYSE STOCK_EXCHANGE
Nasdaq INDEX
S&P INDEX
Nasdaq INDEX


In [58]:
for ents in doc.ents:
  print(ents.text,ents.label_)

Apple COMPANY
Apple COMPANY
AAPL.O STOCK
Apple COMPANY
Nasdaq COMPANY
S&P 500 INDEX
S&P 500 INDEX
ET STOCK
Dow Jones Industrial Average INDEX
S&P 500 INDEX
Nasdaq Composite INDEX
S&P 500 INDEX
JD.com COMPANY
Tencent Music COMPANY
TME.N STOCK
NIO.N STOCK
Kroger COMPANY
KR.N STOCK
NYSE STOCK_EXCHANGE
Nasdaq INDEX
S&P INDEX
Nasdaq INDEX


In [59]:
text2 = '''
Apple Inc. designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories,
and sells a variety of related services. The Company’s products include iPhone, Mac, iPad, and Wearables, Home and Accessories.
iPhone is the Company’s line of smartphones based on its iOS operating system.
Mac is the Company’s line of personal computers based on its macOS operating system.
iPad is the Company’s line of multi-purpose tablets based on its iPadOS operating system.
Wearables, Home and Accessories includes AirPods, Apple TV, Apple Watch, Beats products, HomePod, iPod touch and other Apple-branded and third-party accessories. AirPods are the Company’s wireless headphones that interact with Siri. Apple Watch is the Company’s line of smart watches. Its services include Advertising, AppleCare, Cloud Services, Digital Content and Payment Services.
Its customers are primarily in the consumer, small and mid-sized business, education, enterprise and government markets.
'''

In [60]:
doc2=nlp(text2)
for ents in doc2.ents:
  print(ents.text,ents.label_)

Apple COMPANY
Apple COMPANY
TV STOCK
Apple COMPANY
Apple COMPANY
Apple COMPANY


In [66]:
colors = {
    "COMPANY": "#ffcc00",
    "STOCK": "#00ccff",
    "INDEX": "#ff6699",
    "STOCK_EXCHANGE": "#66ff66"
}

options = {
    "ents": ["COMPANY", "STOCK", "INDEX", "STOCK_EXCHANGE"],
    "colors": colors
}

In [67]:
from spacy import displacy
displacy.render(doc,style="ent",options=options)

In [68]:
displacy.render(doc2,style="ent",options=options)